In [81]:
!pip install sqlalchemy psycopg2-binary

In [82]:
from sqlalchemy import create_engine, text, URL
from getpass import getpass

In [83]:
password = getpass("Введите пароль PostgreSQL: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="TEST"
)

engine = create_engine(connection_url)

try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    print("Успешное подключение к PostgreSQL")
except Exception as error:
    print(f"Ошибка подключения: {error}")

Введите пароль PostgreSQL:  ········


Успешное подключение к PostgreSQL


In [84]:
import pandas as pd

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)

tables

,table_name
0,brand
1,client
2,model
3,order_status
4,order_status_history
5,orders
6,warehouse


In [85]:
query = """
SELECT *
FROM orders
LIMIT 10;
"""

orders_sample = pd.read_sql(query, engine)

orders_sample

,id,order_number,client_id,model_id,ordered_quantity,ordered_at
0,7,007,7,5,50,2026-02-18 22:03:21
1,11,011,1,3,50,2026-02-18 22:03:21
2,14,014,4,12,50,2026-02-18 22:03:21
3,1,001,1,1,50,2026-02-17 22:03:21
4,2,002,2,4,50,2026-02-10 22:03:21
5,3,003,3,7,100,2026-02-01 22:03:21
6,13,013,3,9,100,2026-02-02 22:03:21
7,23,023,3,8,110,2026-02-03 22:03:21
8,4,004,4,10,120,2026-01-18 22:03:21
9,24,024,4,11,120,2026-03-18 22:03:21


In [86]:
query = """
SELECT
    o.order_number,
    o.ordered_at,
    c.client_name,
    m.model_name,
    b.brand_name,
    o.ordered_quantity
FROM orders AS o
JOIN client AS c
    ON o.client_id = c.id
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
ORDER BY o.ordered_at;
"""

orders_analysis = pd.read_sql(query, engine)

orders_analysis.head(10)

,order_number,ordered_at,client_name,model_name,brand_name,ordered_quantity
0,005,2026-01-01 22:03:21,Сидоров С.С.,Acer_X,Acer,200
1,020,2026-01-16 22:03:21,Максимов М.М.,Acer_X,Acer,200
2,004,2026-01-18 22:03:21,Дмитриев Д.Д.,Expert_B,Asus,120
3,003,2026-02-01 22:03:21,Алексеев А.А.,Legion_3,Lenovo,100
4,013,2026-02-02 22:03:21,Алексеев А.А.,Legion_7,Lenovo,100
5,015,2026-02-02 22:03:21,Сидоров С.С.,Acer_Z,Acer,150
6,023,2026-02-03 22:03:21,Алексеев А.А.,Legion_5,Lenovo,110
7,002,2026-02-10 22:03:21,Петров П.П.,Galaxy_S,Samsung,50
8,009,2026-02-10 22:03:21,Андреев А.А.,Asus_70,Asus,120
9,008,2026-02-11 22:03:21,Сергеев С.С.,Legion_5,Lenovo,110


In [87]:
query = text("""
SELECT
    o.order_number,
    o.ordered_at,
    c.client_name,
    m.model_name,
    b.brand_name,
    o.ordered_quantity
FROM orders AS o
JOIN client AS c
    ON o.client_id = c.id
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
WHERE o.ordered_at >= :start_date
ORDER BY o.ordered_at;
""")

orders_from_february = pd.read_sql(
    query,
    engine,
    params={"start_date": "2026-02-01"}
)

orders_from_february.head()

,order_number,ordered_at,client_name,model_name,brand_name,ordered_quantity
0,003,2026-02-01 22:03:21,Алексеев А.А.,Legion_3,Lenovo,100
1,015,2026-02-02 22:03:21,Сидоров С.С.,Acer_Z,Acer,150
2,013,2026-02-02 22:03:21,Алексеев А.А.,Legion_7,Lenovo,100
3,023,2026-02-03 22:03:21,Алексеев А.А.,Legion_5,Lenovo,110
4,002,2026-02-10 22:03:21,Петров П.П.,Galaxy_S,Samsung,50


In [88]:
brand_analysis = (
    orders_from_february
    .groupby('brand_name')
    .agg(
        total_quantity=('ordered_quantity', 'sum'),
        orders=('order_number', 'nunique')
    )
    .sort_values('total_quantity', ascending=False)
)

brand_analysis['share_pct'] = (
    brand_analysis['total_quantity']
    / brand_analysis['total_quantity'].sum()
    * 100
).round(2)

top3_brands = brand_analysis.head(3)

top3_brands

,total_quantity,orders,share_pct
brand_name,,,
Lenovo,520,5,31.42
Asus,410,4,24.77
Acer,250,3,15.11


In [89]:
query = text("""
SELECT
    b.brand_name,
    SUM(o.ordered_quantity) AS total_quantity,
    COUNT(DISTINCT o.order_number) AS orders
FROM orders AS o
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
WHERE o.ordered_at >= :start_date
GROUP BY b.brand_name
ORDER BY total_quantity DESC
LIMIT 3;
""")

top3_sql = pd.read_sql(
    query,
    engine,
    params={"start_date": "2026-02-01"}
)

top3_sql

,brand_name,total_quantity,orders
0,Lenovo,520,5
1,Asus,410,4
2,Acer,250,3


In [90]:
query = text("""
SELECT
    o.order_number,
    c.client_name,
    m.model_name,
    b.brand_name,
    os.status_name,
    osh.status_begins,
    osh.status_ends
FROM order_status_history AS osh
JOIN orders AS o
    ON osh.order_id = o.id
JOIN client AS c
    ON o.client_id = c.id
JOIN model AS m
    ON o.model_id = m.id
JOIN brand AS b
    ON m.brand_id = b.id
JOIN order_status AS os
    ON osh.status_id = os.id
ORDER BY o.order_number, osh.status_begins;
""")

status_history = pd.read_sql(query, engine)

status_history.head(10)

,order_number,client_name,model_name,brand_name,status_name,status_begins,status_ends
0,001,Иванов И.И.,Iphone_17,Apple,delivered,2026-02-18,2026-02-18
1,002,Петров П.П.,Galaxy_S,Samsung,delivered,2026-02-17,2026-02-17
2,003,Алексеев А.А.,Legion_3,Lenovo,delivered,2026-02-16,2026-02-16
3,004,Дмитриев Д.Д.,Expert_B,Asus,in_transit,2026-02-18,None
4,005,Сидоров С.С.,Acer_X,Acer,in_transit,2026-02-17,None
5,006,Николаев Н.Н.,Iphone_16,Apple,paid,2026-02-18,None
6,007,Олегов О.О.,Galaxy_R,Samsung,paid,2026-02-17,None
7,008,Сергеев С.С.,Legion_5,Lenovo,paid,2026-02-16,None
8,009,Андреев А.А.,Asus_70,Asus,not_paid,None,None
9,010,Максимов М.М.,Acer_Y,Acer,not_paid,None,None


In [91]:
status_history['status_begins'] = pd.to_datetime(status_history['status_begins'], errors='coerce')
status_history['status_ends']   = pd.to_datetime(status_history['status_ends'],   errors='coerce')

In [92]:
status_history

,order_number,client_name,model_name,brand_name,status_name,status_begins,status_ends
0,001,Иванов И.И.,Iphone_17,Apple,delivered,2026-02-18,2026-02-18
1,002,Петров П.П.,Galaxy_S,Samsung,delivered,2026-02-17,2026-02-17
2,003,Алексеев А.А.,Legion_3,Lenovo,delivered,2026-02-16,2026-02-16
3,004,Дмитриев Д.Д.,Expert_B,Asus,in_transit,2026-02-18,NaT
4,005,Сидоров С.С.,Acer_X,Acer,in_transit,2026-02-17,NaT
5,006,Николаев Н.Н.,Iphone_16,Apple,paid,2026-02-18,NaT
6,007,Олегов О.О.,Galaxy_R,Samsung,paid,2026-02-17,NaT
7,008,Сергеев С.С.,Legion_5,Lenovo,paid,2026-02-16,NaT
8,009,Андреев А.А.,Asus_70,Asus,not_paid,NaT,NaT
9,010,Максимов М.М.,Acer_Y,Acer,not_paid,NaT,NaT


In [93]:
status_history['duration'] = (
    status_history['status_ends']
    - status_history['status_begins']
)

In [94]:
status_history['duration_days'] = (
    status_history['duration'].dt.days
)

In [95]:
status_history

,order_number,client_name,model_name,brand_name,status_name,status_begins,status_ends,duration,duration_days
0,001,Иванов И.И.,Iphone_17,Apple,delivered,2026-02-18,2026-02-18,0 days,0.0
1,002,Петров П.П.,Galaxy_S,Samsung,delivered,2026-02-17,2026-02-17,0 days,0.0
2,003,Алексеев А.А.,Legion_3,Lenovo,delivered,2026-02-16,2026-02-16,0 days,0.0
3,004,Дмитриев Д.Д.,Expert_B,Asus,in_transit,2026-02-18,NaT,NaT,NaN
4,005,Сидоров С.С.,Acer_X,Acer,in_transit,2026-02-17,NaT,NaT,NaN
5,006,Николаев Н.Н.,Iphone_16,Apple,paid,2026-02-18,NaT,NaT,NaN
6,007,Олегов О.О.,Galaxy_R,Samsung,paid,2026-02-17,NaT,NaT,NaN
7,008,Сергеев С.С.,Legion_5,Lenovo,paid,2026-02-16,NaT,NaT,NaN
8,009,Андреев А.А.,Asus_70,Asus,not_paid,NaT,NaT,NaT,NaN
9,010,Максимов М.М.,Acer_Y,Acer,not_paid,NaT,NaT,NaT,NaN


In [96]:
status_analysis = (
    status_history
    .dropna(subset=['duration_days'])
    .groupby('status_name')
    .agg(
        avg_duration_days=('duration_days', 'mean'),
        completed_intervals=('duration_days', 'count')
    )
    .sort_values('avg_duration_days', ascending=False)
)

status_analysis

,avg_duration_days,completed_intervals
status_name,,
delivered,0.0,3


In [97]:
current_status = (
    status_history
    .sort_values(['order_number', 'status_begins'])
    .groupby('order_number')
    .tail(1)
)

current_status[['order_number', 'client_name', 'model_name', 'brand_name', 'status_name']]

,order_number,client_name,model_name,brand_name,status_name
0,001,Иванов И.И.,Iphone_17,Apple,delivered
1,002,Петров П.П.,Galaxy_S,Samsung,delivered
2,003,Алексеев А.А.,Legion_3,Lenovo,delivered
3,004,Дмитриев Д.Д.,Expert_B,Asus,in_transit
4,005,Сидоров С.С.,Acer_X,Acer,in_transit
5,006,Николаев Н.Н.,Iphone_16,Apple,paid
6,007,Олегов О.О.,Galaxy_R,Samsung,paid
7,008,Сергеев С.С.,Legion_5,Lenovo,paid
8,009,Андреев А.А.,Asus_70,Asus,not_paid
9,010,Максимов М.М.,Acer_Y,Acer,not_paid


In [98]:
status_distribution = (
    current_status
    .groupby('status_name')
    .size()
    .to_frame('orders')
)

status_distribution['share_pct'] = (
    status_distribution['orders']
    / status_distribution['orders'].sum()
    * 100
).round(2)

status_distribution = status_distribution.sort_values(
    'orders',
    ascending=False
)

status_distribution

,orders,share_pct
status_name,,
delivered,3,30.0
paid,3,30.0
in_transit,2,20.0
not_paid,2,20.0


In [99]:
brand_status = (
    current_status
    .pivot_table(
        index='brand_name',
        columns='status_name',
        values='order_number',
        aggfunc='nunique',
        fill_value=0
    )
)

brand_status

status_name,delivered,in_transit,not_paid,paid
brand_name,,,,
Acer,0,1,1,0
Apple,1,0,0,1
Asus,0,1,1,0
Lenovo,1,0,0,1
Samsung,1,0,0,1


In [100]:
dominant_status = brand_status.idxmax(axis=1)

dominant_status

brand_name
Acer       in_transit
Apple       delivered
Asus       in_transit
Lenovo      delivered
Samsung     delivered
dtype: object

In [101]:
status_history

,order_number,client_name,model_name,brand_name,status_name,status_begins,status_ends,duration,duration_days
0,001,Иванов И.И.,Iphone_17,Apple,delivered,2026-02-18,2026-02-18,0 days,0.0
1,002,Петров П.П.,Galaxy_S,Samsung,delivered,2026-02-17,2026-02-17,0 days,0.0
2,003,Алексеев А.А.,Legion_3,Lenovo,delivered,2026-02-16,2026-02-16,0 days,0.0
3,004,Дмитриев Д.Д.,Expert_B,Asus,in_transit,2026-02-18,NaT,NaT,NaN
4,005,Сидоров С.С.,Acer_X,Acer,in_transit,2026-02-17,NaT,NaT,NaN
5,006,Николаев Н.Н.,Iphone_16,Apple,paid,2026-02-18,NaT,NaT,NaN
6,007,Олегов О.О.,Galaxy_R,Samsung,paid,2026-02-17,NaT,NaT,NaN
7,008,Сергеев С.С.,Legion_5,Lenovo,paid,2026-02-16,NaT,NaT,NaN
8,009,Андреев А.А.,Asus_70,Asus,not_paid,NaT,NaT,NaT,NaN
9,010,Максимов М.М.,Acer_Y,Acer,not_paid,NaT,NaT,NaT,NaN


In [102]:
status_history = status_history.sort_values(
    ['order_number', 'status_begins']
).copy()

status_history['prev_status'] = (
    status_history.groupby('order_number')['status_name'].shift(1)
)

status_history['prev_end'] = (
    status_history.groupby('order_number')['status_ends'].shift(1)
)

status_history['gap_days'] = (
    status_history['status_begins'] - status_history['prev_end']
).dt.days

In [103]:
status_history

,order_number,client_name,model_name,brand_name,status_name,status_begins,status_ends,duration,duration_days,prev_status,prev_end,gap_days
0,001,Иванов И.И.,Iphone_17,Apple,delivered,2026-02-18,2026-02-18,0 days,0.0,NaN,NaT,NaN
1,002,Петров П.П.,Galaxy_S,Samsung,delivered,2026-02-17,2026-02-17,0 days,0.0,NaN,NaT,NaN
2,003,Алексеев А.А.,Legion_3,Lenovo,delivered,2026-02-16,2026-02-16,0 days,0.0,NaN,NaT,NaN
3,004,Дмитриев Д.Д.,Expert_B,Asus,in_transit,2026-02-18,NaT,NaT,NaN,NaN,NaT,NaN
4,005,Сидоров С.С.,Acer_X,Acer,in_transit,2026-02-17,NaT,NaT,NaN,NaN,NaT,NaN
5,006,Николаев Н.Н.,Iphone_16,Apple,paid,2026-02-18,NaT,NaT,NaN,NaN,NaT,NaN
6,007,Олегов О.О.,Galaxy_R,Samsung,paid,2026-02-17,NaT,NaT,NaN,NaN,NaT,NaN
7,008,Сергеев С.С.,Legion_5,Lenovo,paid,2026-02-16,NaT,NaT,NaN,NaN,NaT,NaN
8,009,Андреев А.А.,Asus_70,Asus,not_paid,NaT,NaT,NaT,NaN,NaN,NaT,NaN
9,010,Максимов М.М.,Acer_Y,Acer,not_paid,NaT,NaT,NaT,NaN,NaN,NaT,NaN
